In [79]:
from datetime import date
print(dir(date))

['__add__', '__class__', '__delattr__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__ne__', '__new__', '__radd__', '__reduce__', '__reduce_ex__', '__repr__', '__rsub__', '__setattr__', '__sizeof__', '__str__', '__sub__', '__subclasshook__', 'ctime', 'day', 'fromisocalendar', 'fromisoformat', 'fromordinal', 'fromtimestamp', 'isocalendar', 'isoformat', 'isoweekday', 'max', 'min', 'month', 'replace', 'resolution', 'strftime', 'timetuple', 'today', 'toordinal', 'weekday', 'year']


In [77]:
d = date(2023, 10, 5)

In [12]:
print(d.year, d.month, d.day)
print(d.weekday)
print(d)
print(d.isoformat())
print(type(d))
print(type(d.isoformat()))

2023 10 5
<built-in method weekday of datetime.date object at 0x00000273BC9819D0>
2023-10-05
2023-10-05
<class 'datetime.date'>
<class 'str'>


In [80]:
from datetime import timedelta
print(dir(timedelta))

['__abs__', '__add__', '__bool__', '__class__', '__delattr__', '__dir__', '__divmod__', '__doc__', '__eq__', '__floordiv__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__mod__', '__mul__', '__ne__', '__neg__', '__new__', '__pos__', '__radd__', '__rdivmod__', '__reduce__', '__reduce_ex__', '__repr__', '__rfloordiv__', '__rmod__', '__rmul__', '__rsub__', '__rtruediv__', '__setattr__', '__sizeof__', '__str__', '__sub__', '__subclasshook__', '__truediv__', 'days', 'max', 'microseconds', 'min', 'resolution', 'seconds', 'total_seconds']


In [14]:
timedelta(days=5, 
          hours=3,
            minutes=30, 
            seconds=15,
              milliseconds=500, 
              microseconds=250)

datetime.timedelta(days=5, seconds=12615, microseconds=500250)

In [15]:
d + timedelta(days=35)

datetime.date(2023, 11, 9)

In [81]:
def days_in_month(year, month):
    if month == 12:
        return 31
    return (date(year, month + 1, 1) - timedelta(days=1)).day

In [17]:
days_in_month(2022, 2)

28

In [82]:
def add_months(d, n):
    total = d.month - 1 + n
    year = d.year + total // 12
    month = total % 12 + 1
    day = min(d.day, days_in_month(year, month))
    return date(year, month, day)

In [19]:
add_months(d, 0)

datetime.date(2023, 10, 5)

In [83]:
holidays = set()
for s in ["2026-01-01", "2026-12-25"]:
    holidays.add(date.fromisoformat(s))

holidays_banks = set()
for s in ["2026-02-28", "2026-03-12"]:
    holidays_banks.add(date.fromisoformat(s))

In [84]:
recurring = {(1, 1), (12, 25)}

def is_business_day(d, holidays = None , recurring = None):
    if recurring is None:
        recurring = set()
    if holidays is None:
        holidays = set()
    if d.weekday() >= 5:
        return False
    if d in holidays:
        return False
    if (d.month, d.day) in recurring:
        return False
    return True

In [24]:
is_business_day(d, holidays)

True

Here we define the adjustinng conventions:
- Following convention
- Modified Following
- Preceding

In [85]:
def next_business_day(d, holidays, recurring):
    while not is_business_day(d, holidays, recurring):
        d = d + timedelta(days=1)
    return d

In [48]:
d = date(2026, 8, 22)

In [ ]:
is_business_day(d, holidays, set())

False

In [50]:
next_business_day(d, holidays)

datetime.date(2026, 8, 24)

In [86]:
def prev_business_day(d, holidays, recurring):
    while not is_business_day(d, holidays, recurring):
        d = d - timedelta(days=1)
    return d

In [ ]:
prev_business_day(d, holidays, set())

datetime.date(2026, 8, 21)

In [87]:
def adjust(d, convention, holidays, recurring):
    if convention == "FOLLOWING":
        return next_business_day(d, holidays, recurring)
    if convention == "PRECEDING":
        return prev_business_day(d, holidays, recurring)
    if convention == "MODIFIED_FOLLOWING":
        candidate = next_business_day(d, holidays, recurring)
        if candidate.month != d.month:
            return prev_business_day(d, holidays, recurring)
        return candidate
    raise ValueError("unknown convention: " + convention)

In [67]:
adjust(d, "MODIFIED FOLLOWING", holidays)

ValueError: unknown convention: MODIFIED FOLLOWING

esto me gustaria entenderlo

In [ ]:
from enum import Enum

class Convention(Enum):
    FOLLOWING = "FOLLOWING"
    MODIFIED_FOLLOWING = "MODIFIED_FOLLOWING"
    PRECEDING = "PRECEDING"

In [57]:
Convention.FOLLOWING 

<Convention.FOLLOWING: 'FOLLOWING'>

In [88]:
def parse_frequency(text):
    text = text.strip().upper()
    number, unit = text[:-1], text[-1]

    if not number.isdigit():
        raise ValueError(f"invalid frequency: {text}")

    number = int(number)
    if number <= 0:
        raise ValueError(f"frequency must be positive: {text}")

    if unit == "M":
        return number
    if unit == "Y":
        return number * 12
    raise ValueError(f"unsupported frequency unit: {unit}")

In [74]:
parse_frequency("1Y")

12

In [89]:
def generate_unadjusted(trade, maturity, months, generation="BACKWARD"):
    if maturity <= trade:
        raise ValueError("maturity must be after trade date")

    if generation == "BACKWARD":
        return _generate_backward(trade, maturity, months)
    if generation == "FORWARD":
        return _generate_forward(trade, maturity, months)
    raise ValueError(f"unknown generation rule: {generation}")


def _generate_backward(trade, maturity, months):
    dates = []
    n = 0
    while True:
        d = add_months(maturity, -months * n)
        if d <= trade:
            break
        dates.append(d)
        n += 1
    dates.reverse() # just to order the dates from trade to maturity
    return dates


def _generate_forward(trade, maturity, months):
    dates = []
    n = 1
    while True:
        d = add_months(trade, months * n)
        if d >= maturity:
            break
        dates.append(d)
        n += 1
    dates.append(maturity)
    return dates

In [90]:
def test_stub_position_differs_by_generation():
    trade, maturity = date(2026, 1, 15), date(2027, 3, 15)
    back = generate_unadjusted(trade, maturity, 6, "BACKWARD")
    fwd = generate_unadjusted(trade, maturity, 6, "FORWARD")
    assert len(back) == len(fwd) == 3
    assert back[0] == date(2026, 3, 15)
    assert fwd[0] == date(2026, 7, 15)

In [36]:
test_stub_position_differs_by_generation()
# print("ok")

In [91]:
def build_schedule(trade, maturity, frequency, convention, holidays,
                   recurring=None, generation="BACKWARD"):
    months = parse_frequency(frequency)
    unadjusted = generate_unadjusted(trade, maturity, months, generation)
    adjusted = [adjust(d, convention, holidays, recurring) for d in unadjusted]
    return unadjusted, adjusted

In [96]:
# Test 1

def test_business_day_is_unchanged():
    d = date(2026, 8, 19)
    result = next_business_day(d, set(), set())
    assert result == d

In [101]:
test_business_day_is_unchanged()
print("ok")

ok


In [45]:
# Test 2
def test_saturday_moves_forward():
    d = date(2026, 8, 15)
    result = next_business_day(d, set(), set())
    assert result == date(2026, 8, 17)

In [95]:
test_saturday_moves_forward()
print("ok")

ok


In [93]:
# Test 3
def test_saturday_moves_backward():
    d = date(2026, 8, 15)
    result = prev_business_day(d, set(), set())
    assert result == date(2026, 8, 14)

In [102]:
test_saturday_moves_backward()
print("ok")

ok


In [104]:
# test 4
def test_holday_skipped():
    d = date (2026, 8, 19)
    holidays = {date (2026, 8, 19)}
    result = adjust(d, "FOLLOWING", holidays, set())
    assert result == date (2026, 8, 20)

In [105]:
test_holday_skipped()
print ("ok")

ok


In [107]:
# test 5 
def test_modified_following__end_month ():
    d = date (2026, 5, 30)
    result = adjust (d, "MODIFIED_FOLLOWING", set(), set())
    assert result == date(2026, 5, 29)

In [109]:
test_modified_following__end_month()
print("ok")

ok


In [110]:
# test 6
def test_modified_following__middle_month ():
    d = date (2026, 8, 15)
    result = adjust (d, "MODIFIED_FOLLOWING", set(), set())
    assert result == date(2026, 8, 17)

In [111]:
test_modified_following__middle_month()
print("ok")

ok


In [112]:
# test 7
def test_weekend_and_holiday_and_recurring ():
    d = date (2026, 8, 22)
    holidays = {date(2026, 8, 24)}
    recurring = {(8, 25)}
    result = adjust (d, "MODIFIED_FOLLOWING", holidays, recurring)
    assert result == date(2026, 8, 26)

In [113]:
test_weekend_and_holiday_and_recurring()
print("ok")

ok


In [114]:
# test 8
def test_parse_frequency():
    assert parse_frequency("1M")== 1
    assert parse_frequency ("1Y") == 12

In [115]:
test_parse_frequency()
print("ok")

ok


In [118]:
# Test 9
def test_unadjusted():
    result = generate_unadjusted(date(2026, 8, 19), date(2027, 8, 22), 6, generation="BACKWARD")
    assert result == [date(2026, 8, 22), date(2027, 2, 22), date(2027, 8, 22)]

In [119]:
test_unadjusted()
print("ok")

ok


In [66]:
# test 10
def test_trade_not_payment_date():
    trade = date(2026, 8, 19)
    dates = generate_unadjusted (trade, date(2027, 8, 19), 6)
    assert trade not in dates

In [67]:
test_trade_not_payment_date
print("ok")

ok


In [68]:
# test 11

def test_build_schedule_returns_both_lists():
    unadjusted, adjusted = build_schedule(
        date(2026, 1, 15), date(2027, 1, 15), "6M",
        "MODIFIED_FOLLOWING", set(), set()
    )
    assert len(unadjusted) == len(adjusted) == 2
    for u, a in zip(unadjusted, adjusted):
        assert abs((a - u).days) <= 4

In [69]:
test_build_schedule_returns_both_lists
print("ok")

ok


In [70]:
# test 12

def test_build_schedule_applies_holidays():
    unadjusted, adjusted = build_schedule(
        date(2026, 1, 15), date(2027, 1, 15), "6M",
        "FOLLOWING", {date(2027, 1, 15)}, set()
    )
    assert unadjusted[-1] == date(2027, 1, 15)
    assert adjusted[-1] == date(2027, 1, 18)

In [71]:
test_build_schedule_applies_holidays
print("ok")

ok


In [ ]:
def test_combined_holiday_calendars():
    calendar_dk = {date(2026, 6, 5)}   
    calendar_us = {date(2026, 7, 3)}     
    combined = calendar_dk | calendar_us

    assert not is_business_day(date(2026, 6, 5), combined, set())
    assert not is_business_day(date(2026, 7, 3), combined, set())
    assert is_business_day(date(2026, 6, 4), combined, set())

In [73]:
test_combined_holiday_calendars
print ("ok")

ok
